# Impact of model discrepancy and noisy values on the classification

In [1]:
from typing import Callable
from types import SimpleNamespace
from pipe import select

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy import linalg

import plotly.express as px
import plotly.graph_objects as go

from rich.progress import track

import warnings
warnings.filterwarnings('ignore', category=np.exceptions.ComplexWarning)

In [2]:
SEED = 123
np.random.seed(SEED)

We will take damped pendulum system as an example of the ground-truth model. Its phase space is 2-dimensional. Its coordinates $(x, y)$ are angle and angular velocity. We will allow the velocity to suddenly increase (under some external forces) in some points of time. We will call it jump (like jump processes in stochastic processes). Then the system will evolve as usual till the next jump. We will demonstrate that the trajectory segmentation will allow to classify the trajectories while the straightforward method will fail.

In [3]:
# system params
systems = [
    {
        "gamma": -0.7,
        "omega": -0.6,
        "u": np.array([1., 0]),
        "v": np.array([0., 1.]),
    },
    {
        "gamma": -0.8,
        "omega": -0.6,
        "u": np.array([1., 0]),
        "v": np.array([0., 1.]),
    }
]
systems = list(systems | select(lambda d: SimpleNamespace(**d)))

In [4]:
# building matrix A in our ODE x' = A @ x
def system_matrix(
    sys_c: SimpleNamespace
):
    L = np.diag(
        [-sys_c.gamma + 1j*sys_c.omega, -sys_c.gamma - 1j*sys_c.omega]
    )
    e1, e2 = list(
        [sys_c.u + 1j*sys_c.v, sys_c.u - 1j*sys_c.v] |
        select(lambda e: e / linalg.norm(e)) |
        select(lambda e: e[:, None])
    )
    U = np.hstack([e1, e2])
    A = (U @ L @ U.T.conj()).astype(float)
    return A

In [5]:
# exact solution
def exact_solver(t: np.ndarray, sys_c: SimpleNamespace, x0: np.ndarray):
    num_points = t.size
    # amplitude
    ampl = linalg.norm(x0)
    # inital phase
    if x0[0] > 0:
        phase = np.arctan(x0[1] / x0[0])
    elif x0[0] < 0:
        phase = np.pi + np.arctan(x0[1] / x0[0])
    else:
        phase = np.sign(x0[1]) * np.pi / 2

    u_rep, v_rep = list(
        [sys_c.u, sys_c.v] |
        select(lambda x: np.repeat(x[:, None], num_points, axis=1))
    )    
    sol = ampl * np.exp(-sys_c.gamma * t) * \
        (np.cos(phase - sys_c.omega * t) * u_rep + np.sin(phase - sys_c.omega * t) * v_rep)
    return sol.T

In [6]:
# numerical solution
def RK_solver(t: np.ndarray, sys_c: SimpleNamespace, x0: np.ndarray):
    A = system_matrix(sys_c)
    return solve_ivp(
        lambda t, x: A.dot(x),
        [t[0], t[-1]],
        x0,
        method="RK23",
        t_eval=t
    ).y.T

In [7]:

def compute_systems_matrices():
    sys_matrices = list(systems | select(lambda d: system_matrix(d)))
    print(*sys_matrices, sep='\n')
compute_systems_matrices()

[[ 0.7 -0.6]
 [ 0.6  0.7]]
[[ 0.8 -0.6]
 [ 0.6  0.8]]


In [8]:
x0 = np.array([1. ,1.])

In [9]:
T = 5.
delta_t = 0.5e-1
t = np.linspace(0., T, int(T / delta_t))
print(t.size)

100


## Solutions without jumps

In [10]:
def solutions_without_jumps():
    # compute exact solutions
    true_sol = list(
        systems |
        select(lambda d: exact_solver(t, d, x0))
    )
    # solve sytems with RK23
    RK23_sol = list(
        systems |
        select(lambda s: RK_solver(t, s, x0))
    )
    solutions = {
        "true": true_sol,
        "RK23": RK23_sol
    }

    fig = go.Figure()
    for sol_name, sol in solutions.items():
        for i, sys_sol in enumerate(sol):
            line = dict(dash="dash") if sol_name.count("true") == 0 else None
            fig.add_trace(
                go.Scatter(
                    x=sys_sol[:, 0],
                    y=sys_sol[:, 1],
                    mode='lines',
                    name=f"{sol_name}_{i}",
                    line=line
                )
            )
    fig.add_trace(
        go.Scatter(
            x=[x0[0]],
            y=[x0[1]],
            name="start"
        )
    )
    fig.show()
    
    return fig

solutions_figure = solutions_without_jumps()

## Introduce jumps to the model

Jumps are shocks which suddenly increase the vecolcity

In [11]:
# amplitude of the juimp
jump_ampl = 0.22
# take 3 jump points uniformly
num_jumps = 10
jump_indx = np.arange(1, num_jumps + 1) * (len(t) // (num_jumps + 2))
#jump_indx = jump_indx[:-3]
print("Jump indices: ", jump_indx)
print("Jump times: ", t[jump_indx])

Jump indices:  [ 8 16 24 32 40 48 56 64 72 80]
Jump times:  [0.4040404  0.80808081 1.21212121 1.61616162 2.02020202 2.42424242
 2.82828283 3.23232323 3.63636364 4.04040404]


In [12]:
def jumped_exact_solver(
    t: np.ndarray, sys_c: SimpleNamespace, x0: np.ndarray, 
    jump_indxs: np.ndarray, jump_ampl: float
):
    t = t.copy()
    jump_indxs = jump_indxs

    cur_x0 = x0
    cur_start = 0
    sol = []
    for jump_indx in jump_indxs:
        cur_t = t[cur_start:jump_indx + 1] - t[cur_start]
        cur_sol = exact_solver(cur_t, sys_c, cur_x0)

        sol.append(cur_sol[:-1])
        # recompute jumped x0
        cur_x0 = cur_sol[-1].copy()
        cur_x0[1] += jump_ampl
        # adjust time for next computation
        cur_start = jump_indx
    
    # append solution for the last segment
    cur_t = t[cur_start:] - t[cur_start]
    sol.append(exact_solver(cur_t, sys_c, cur_x0))

    return np.concat(sol)

In [13]:
# compute exact solution for system 0
jumped_solution_0 = jumped_exact_solver(t, systems[0], x0, jump_indx, jump_ampl)

In [14]:
solutions_figure.add_trace(
    go.Scatter(
        x=jumped_solution_0[:, 0],
        y=jumped_solution_0[:, 1],
        mode='lines',
        name="jumped_0"
    )
)
solutions_figure.show()

## Perform classification with jumps

We will sample significantly noised trajectory of the system 0. Our classifier is the bayessian classifier that chooses model with the least MSE loss

In [15]:
NUM_SAMPLES = 500
sigma = 5
num_traj_points = jumped_solution_0.shape[0]

In [16]:
# we choose the model with the least mse
def cls_loss(sample_traj: np.ndarray, mean_traj: np.ndarray):
    return np.linalg.norm(sample_traj - mean_traj, ord=2, axis=1).mean()

In [17]:
def simple_classification():
    np.random.seed(SEED)
    # compute exact solutions
    true_sol = list(
        systems |
        select(lambda d: exact_solver(t, d, x0))
    )
    # solve sytems with RK23
    RK23_sol = list(
        systems |
        select(lambda s: RK_solver(t, s, x0))
    )
    solutions = {
        "true": true_sol,
        "RK23": RK23_sol
    }

    accuracy_df = pd.DataFrame(
        {
            "accuracy": np.zeros([len(solutions)]),
            "method": ["true_simple", "RK23_simple"]
        }
    )
    for i in range(NUM_SAMPLES):
        sample_traj = jumped_solution_0 + sigma * np.random.randn(num_traj_points, 2)
        
        if i == 0:
            fig = go.Figure()
            fig.add_trace(
                go.Scatter(
                    x=jumped_solution_0[:, 0],
                    y=jumped_solution_0[:, 1],
                    mode='lines',
                    name="jumped_0"
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=sample_traj[:, 0],
                    y=sample_traj[:, 1],
                    mode='markers',
                    name="jumped_sample"
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=true_sol[0][:, 0],
                    y=true_sol[0][:, 1],
                    mode='lines',
                    name="segment0_true"
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=RK23_sol[0][:, 0],
                    y=RK23_sol[0][:, 1],
                    mode='lines',
                    name="segment0_rk",
                    line=dict(dash="dash")
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=true_sol[1][:, 0],
                    y=true_sol[1][:, 1],
                    mode='lines',
                    name="segment1_true"
                )
            )
            fig.show()

        for sol_name, sol in solutions.items():
            sol_losses = list(
                range(2) |
                select(lambda sys_id: cls_loss(sample_traj, sol[sys_id]))
            )
            if sol_losses[0] < sol_losses[1]:
                accuracy_df.loc[
                    accuracy_df["method"] == sol_name, 
                    "accuracy"
                ] += 1.
    accuracy_df.loc[:, "accuracy"] /= NUM_SAMPLES

    return accuracy_df

accuracy_df = simple_classification()
accuracy_df

,accuracy,method
0,0.0,true_simple
1,0.0,RK23_simple


## Perfrom classification with trajectory segmenting

In [18]:
accuracy_df["method"] = ["true", "RK23"]
accuracy_df["segment_size"] = t.size

In [19]:
def segmented_solution(
    t: np.ndarray, sys_c: SimpleNamespace, traj: np.ndarray, 
    segment_size: int, solver_f: Callable
):
    """The trajectory is split into segments of length segment_size.
        Each segment is integrated separetely. The overall loss is a sum of
        the losses over the segments. The initial point of each segment is taken
        as it is, we do not account for small noise here.
    """
    sol = []
    cur_start = 0
    while cur_start < t.size:
        cur_x0 = traj[cur_start]
        sol.append(
            solver_f(
                # x0 is included in the segment length
                t[cur_start: min(cur_start + segment_size, t.size)] - t[cur_start],
                sys_c,
                cur_x0
            )
        )
        cur_start = cur_start + segment_size

    return np.concat(sol)

In [20]:
segment_sizes = np.array([30, 20, 10, 5])

In [21]:
def segmented_classification():
    accuracy_df = []
    for seg_size in track(segment_sizes, "Segment size"):
        np.random.seed(SEED)
        cur_accuracy_df = pd.DataFrame(
            {
                "accuracy": np.zeros([2]),
                "method": [f"true", f"RK23"],
                "segment_size": np.full([2], seg_size)
            }
        )
        for i in track(range(NUM_SAMPLES), "Trajectory num", transient=True):
            sample_traj = jumped_solution_0 + sigma * np.random.randn(num_traj_points, 2)

            true_seg_solutions = [
                segmented_solution(t, systems[0], sample_traj, seg_size, exact_solver),
                exact_solver(t, systems[1], x0)
            ]
            rk_seg_solutions = [
                segmented_solution(t, systems[0], sample_traj, seg_size, RK_solver),
                RK_solver(t, systems[1], x0)
            ]
            solutions = {
                f"true": true_seg_solutions,
                f"RK23": rk_seg_solutions
            }

            if i == 0:
                print("Segment size =", seg_size)
                fig = go.Figure()
                fig.add_trace(
                    go.Scatter(
                        x=jumped_solution_0[:, 0],
                        y=jumped_solution_0[:, 1],
                        mode='lines',
                        name="jumped_0"
                    )
                )
                fig.add_trace(
                    go.Scatter(
                        x=sample_traj[:, 0],
                        y=sample_traj[:, 1],
                        mode='markers',
                        name="jumped_sample"
                    )
                )
                fig.add_trace(
                    go.Scatter(
                        x=true_seg_solutions[0][:, 0],
                        y=true_seg_solutions[0][:, 1],
                        mode='lines',
                        name="segment0_true"
                    )
                )
                fig.add_trace(
                    go.Scatter(
                        x=rk_seg_solutions[0][:, 0],
                        y=rk_seg_solutions[0][:, 1],
                        mode='lines',
                        name="segment0_rk",
                        line=dict(dash="dash")
                    )
                )
                fig.add_trace(
                    go.Scatter(
                        x=true_seg_solutions[1][:, 0],
                        y=true_seg_solutions[1][:, 1],
                        mode='lines',
                        name="segment1_true"
                    )
                )
                fig.show()

            for sol_name, sol in solutions.items():
                sol_losses = list(
                    range(2) |
                    select(lambda sys_id: cls_loss(sample_traj, sol[sys_id]))
                )
                if sol_losses[0] < sol_losses[1]:
                    cur_accuracy_df.loc[
                        cur_accuracy_df["method"] == sol_name, 
                        "accuracy"
                    ] += 1.
        cur_accuracy_df.loc[:, "accuracy"] /= NUM_SAMPLES
        accuracy_df.append(cur_accuracy_df)

    accuracy_df = pd.concat(accuracy_df)
    return accuracy_df

accuracy_df = pd.concat([accuracy_df, segmented_classification()])
accuracy_df

Output()

Segment size = 30

Segment size = 20

Segment size = 10

Segment size = 5

,accuracy,method,segment_size
0,0.000,true,100
1,0.000,RK23,100
0,0.000,true,30
1,0.000,RK23,30
0,0.002,true,20
1,0.002,RK23,20
0,0.002,true,10
1,0.002,RK23,10
0,0.116,true,5
1,0.124,RK23,5


## Perform classification with trajectory segmenting and $x_0$ optimizing

In [22]:
from scipy import optimize

def segmented_solution_adv(
    t: np.ndarray, sys_c: SimpleNamespace, traj: np.ndarray, 
    segment_size: int, solver_f: Callable
):
    sol = []
    cur_start = 0
    while cur_start < t.size:
        def obj(x0: np.ndarray):
            pred = solver_f(
                # x0 is included in the segment length
                t[cur_start: min(cur_start + segment_size, t.size)] - t[cur_start],
                sys_c,
                x0
            )
            true = traj[cur_start: min(cur_start + segment_size, t.size)]
            return cls_loss(true, pred)
        # initial guess for true x0
        cur_x0 = traj[cur_start]
        cur_x0 = optimize.minimize(obj, cur_x0, method="Nelder-Mead").x

        sol.append(
            solver_f(
                # x0 is included in the segment length
                t[cur_start: min(cur_start + segment_size, t.size)] - t[cur_start],
                sys_c,
                cur_x0
            )
        )
        cur_start = cur_start + segment_size

    return np.concat(sol)

In [23]:
def segmented_classification_adv():
    accuracy_df = []
    # DEBUG
    # segment_sizes = [30, 20]
    for seg_size in track(segment_sizes, "Segment size"):
        np.random.seed(SEED)
        cur_accuracy_df = pd.DataFrame(
            {
                "accuracy": np.zeros([2]),
                "method": ["true_adv", "RK23_adv"],
                "segment_size": np.full([2], seg_size)
            }
        )
        for i in track(range(NUM_SAMPLES), "Trajectory num", transient=True):
            sample_traj = jumped_solution_0 + sigma * np.random.randn(num_traj_points, 2)

            true_seg_solutions = [
                segmented_solution_adv(t, systems[0], sample_traj, seg_size, exact_solver),
                exact_solver(t, systems[1], x0)
            ]
            rk_seg_solutions = [
                segmented_solution_adv(t, systems[0], sample_traj, seg_size, RK_solver),
                RK_solver(t, systems[1], x0)
            ]
            solutions = {
                "true_adv": true_seg_solutions,
                "RK23_adv": rk_seg_solutions
            }

            if i == 0:
                print("Segment size =", seg_size)
                fig = go.Figure()
                fig.add_trace(
                    go.Scatter(
                        x=jumped_solution_0[:, 0],
                        y=jumped_solution_0[:, 1],
                        mode='lines',
                        name="jumped_0"
                    )
                )
                fig.add_trace(
                    go.Scatter(
                        x=sample_traj[:, 0],
                        y=sample_traj[:, 1],
                        mode='markers',
                        name="jumped_sample"
                    )
                )
                fig.add_trace(
                    go.Scatter(
                        x=true_seg_solutions[0][:, 0],
                        y=true_seg_solutions[0][:, 1],
                        mode='lines',
                        name="segment0_true"
                    )
                )
                fig.add_trace(
                    go.Scatter(
                        x=rk_seg_solutions[0][:, 0],
                        y=rk_seg_solutions[0][:, 1],
                        mode='lines',
                        name="segment0_rk",
                        line=dict(dash="dash")
                    )
                )
                fig.add_trace(
                    go.Scatter(
                        x=true_seg_solutions[1][:, 0],
                        y=true_seg_solutions[1][:, 1],
                        mode='lines',
                        name="segment1_true"
                    )
                )
                fig.show()

            for sol_name, sol in solutions.items():
                sol_losses = list(
                    range(2) |
                    select(lambda sys_id: cls_loss(sample_traj, sol[sys_id]))
                )
                if sol_losses[0] < sol_losses[1]:
                    cur_accuracy_df.loc[
                        cur_accuracy_df["method"] == sol_name, 
                        "accuracy"
                    ] += 1.
        cur_accuracy_df.loc[:, "accuracy"] /= NUM_SAMPLES
        accuracy_df.append(cur_accuracy_df)

    accuracy_df = pd.concat(accuracy_df)
    return accuracy_df

accuracy_df = pd.concat([accuracy_df, segmented_classification_adv()])
accuracy_df

Output()

Segment size = 30

Segment size = 20

Segment size = 10

Segment size = 5

,accuracy,method,segment_size
0,0.000,true,100
1,0.000,RK23,100
0,0.000,true,30
1,0.000,RK23,30
0,0.002,true,20
1,0.002,RK23,20
0,0.002,true,10
1,0.002,RK23,10
0,0.116,true,5
1,0.124,RK23,5


## Vizualize the results

In [24]:
px.bar(
    accuracy_df,
    x="segment_size",
    y="accuracy",
    color="method",
    barmode='group'
)